# Customer Retention Modeling Using Machine Learning

## Data exploration

I began by exploring the dataset structure to understand feature types, missing values, and overall dimensionality. The dataset contained 10,000 observations and both numerical and categorical variables. During initial inspection, I identified missing values in the Tenure column (~9%), which required preprocessing before model training.

In [1]:
# Data manipulation and dataframe operations
import pandas as pd
# Numerical computing
import numpy as np
# Categorical feature encoding
from sklearn.preprocessing import OrdinalEncoder
# Train-test data splitting
from sklearn.model_selection import train_test_split
# Ensemble learning (Random Forest classifier)
from sklearn.ensemble import RandomForestClassifier
# Logistic Regression classifier
from sklearn.linear_model import LogisticRegression
# ROC-AUC evaluation metric
from sklearn.metrics import roc_auc_score
# F1-score evaluation metric
from sklearn.metrics import f1_score
# Dataset shuffling for resampling techniques
from sklearn.utils import shuffle

In [2]:
# Load dataset
df = pd.read_csv('../data/bank_churn.csv')')

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/Churn.csv'

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['Tenure']

In [ ]:
df['Tenure'].max()

In [ ]:
df['Tenure'].min()

In [ ]:
df['Tenure'].mean()

In [ ]:
df['Tenure'].median()

In [ ]:
df['Tenure'].unique()

Tenure presented approximately 9% missing values. Since its distribution was approximately symmetric (mean ≈ median), I applied median imputation to preserve robustness against potential outliers. After imputation, I converted the feature to integer format to maintain consistency with its discrete nature.

In [ ]:
df['Tenure'].fillna(df['Tenure'].median(), inplace = True)

In [ ]:
df['Tenure'] = df['Tenure'].astype(int)

Verificamos los resultados

In [ ]:
df['Tenure'].isnull().sum()

In [ ]:
df['Tenure']

We will first remove the columns that do not provide relevant information for the study. This step ensures that the dataset remains focused only on variables that contribute to the research objectives and avoids unnecessary complexity during the merging and modeling stages.

After this initial filtering process, we will proceed to review each remaining column individually, from top to bottom, to assess its relevance, completeness, and consistency. During this review, we will evaluate whether each variable contributes meaningful information to the analysis or whether it should be excluded to improve data quality and model performance.

In [ ]:
df.info()

I removed non-informative identifiers such as RowNumber and Surname, as they do not provide predictive signal and may introduce noise. Customer identifiers were excluded from modeling to avoid leakage or artificial correlations.

In [ ]:
#Review dataset after feature removal
df.drop(['RowNumber','Surname'], axis = 1, inplace = True) 

In [ ]:
df.info()

Categorical variables (Geography and Gender) were encoded using Ordinal Encoding to convert them into numerical format compatible with scikit-learn models. Given the limited number of categories, this encoding preserved computational efficiency while enabling tree-based modeling.

In [ ]:
#Initialize encoder
encoder = OrdinalEncoder()
#Select categorical columns for transformation
columnas = ['Geography', 'Gender']
#Fit and transform categorical features
data_ordinal = pd.DataFrame(encoder.fit_transform(df[columnas]), columns=df[columnas].columns)

In [ ]:
data_ordinal

In [ ]:
df[['Geography', 'Gender']] = data_ordinal

In [ ]:
df

## Class distribution analysis

In [ ]:
df_target = df['Exited']
target_zeros = df_target[df_target==0]
target_ones = df_target[df_target==1]

values = 0

In [ ]:
target_zeros.shape

values = 1

In [ ]:
target_ones.shape

Class distribution analysis revealed a significant imbalance (approximately 80% non-churn vs 20% churn). Because accuracy can be misleading in imbalanced classification problems, I selected F1-score as the primary optimization metric.

### The dataset was split into training, validation, and test sets to ensure unbiased performance estimation. The validation set was used for hyperparameter tuning, while the test set remained untouched until final evaluation.

In [ ]:
#Separate features and target variable
features = df.drop(['Exited'], axis = 1)
target = df['Exited']

In [ ]:
#First split: training and temporary dataset
features_train, features_temp, target_train, target_temp = train_test_split(features, target, test_size= 0.4, random_state=12345)

In [ ]:
#Second split: validation and test datasets
features_valid, features_test, target_valid, target_test = train_test_split(features_temp, target_temp, test_size= 0.5, random_state=12345)

### Initial modeling without imbalance correction achieved an F1-score close to the target threshold (0.584), suggesting that the dataset contains strong predictive structure. However, minority class recall remained suboptimal.

#### RandomForestClassifier

In [ ]:
'''Iniciamos tres contadores, uno para el número de árboles, para la profundidad de 
los árboles y para la exactitud'''
def BestF1AndAucRocRFC(features_train, target_train, features_valid, target_valid, weight = None):
    best_f1 = 0
    best_depth = 0
    best_est = 0
    best_aucroc = 0
    best_depth2 = 0
    best_est2 = 0
#Contador para el número de árboles
    for trees in range(10,51,10):
    #Profundidad de los árboles
        for depth in range(1,20):
        #Creamos el modelo con sus hiperparámetros
            model = RandomForestClassifier(random_state=12345, n_estimators=trees, max_depth=depth, class_weight= weight)
        #Lo entrenamos
            model.fit(features_train, target_train)
        #Predecimos los valores
            predicted_valid = model.predict(features_valid)
        #Calculamos f1 y aucroc
            f1 = f1_score(target_valid, predicted_valid)
            aucroc = roc_auc_score(target_valid, predicted_valid)
        #Recuperamos la mejor exactitud y la mejor profundidad de árbol y # de árboles
            if f1 > best_f1:
                best_f1 = f1
                best_depth = depth
                best_est = trees
            if aucroc > best_aucroc:
                best_aucroc = aucroc
                best_depth2 = depth
                best_est2 = trees
    print(f'Best F1: Depth = {best_depth} | Trees = {best_est} | F1 = {best_f1:.3f}')
    print(f'Best AUC-ROC: Depth = {best_depth2} | Trees = {best_est2} | AUC-ROC = {best_aucroc:.3f}')

In [ ]:
BestF1AndAucRocRFC(features_train, target_train, features_valid, target_valid)

#### LogisticRegression

In [ ]:
def BestF1AndAucRoc(features_train, target_train, features_valid, target_valid, weight = None):
    best_f1 = 0
    best_threshold = 0
    best_threshold2 = 0
    best_aucroc = 0
    model = LogisticRegression(random_state=12345, solver='liblinear', class_weight = weight)
    model.fit(features_train, target_train)
    probabilities_valid = model.predict_proba(features_valid)
    probabilities_one_valid = probabilities_valid[:, 1]
    for threshold in np.arange(0, 0.3, 0.02):
        predicted_valid = probabilities_one_valid > threshold
        f1 = f1_score(target_valid, predicted_valid)
        aucroc = roc_auc_score(target_valid, predicted_valid)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
        if aucroc > best_aucroc:
            best_aucroc = aucroc
            best_threshold2 = threshold
    print('Best F1: Threshold = {:.2f} | F1 = {:.3f}'.format(best_threshold, best_f1))
    print('Best Aucroc: Threshold = {:.2f} | auc_roc = {:.3f}'.format(best_threshold2, best_aucroc))

In [ ]:
model = LogisticRegression(random_state=12345, solver='liblinear')
model.fit(features_train, target_train)
predicted_valid = model.predict_proba(features_valid)
predicted_valid

In [ ]:
probabilities_one_valid = predicted_valid[:, 1]
auc_roc = roc_auc_score(target_valid, probabilities_one_valid)
auc_roc

In [ ]:
BestF1AndAucRoc(features_train, target_train, features_valid, target_valid)

Logistic Regression showed limited performance under class imbalance. I experimented with probability threshold adjustment to increase sensitivity toward the minority class, but improvements remained marginal compared to tree-based methods.

## To address class imbalance, I implemented two corrective strategies: Class weighting (class_weight='balanced')and Oversampling of the minority class

### Class weighting

#### RandomForestClassifier 


In [ ]:
BestF1AndAucRocRFC(features_train, target_train, features_valid, target_valid, weight = 'balanced')

#### Logistic Regression

In [ ]:
BestF1AndAucRoc(features_train, target_train, features_valid, target_valid, weight = 'balanced')

### Oversampling of the minority class

In [ ]:
def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]

    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)

    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345
    )
    return features_upsampled, target_upsampled

#### RandomForestClassifier 

In [ ]:
for index in range(1,4):
    print(f'Cuando usamos {index}:')
    features_upsampled, target_upsampled = upsample(features_train, target_train, index)
    BestF1AndAucRocRFC(features_upsampled, target_upsampled, features_valid, target_valid)

#### Logistic Regression

In [ ]:
for index in range(1,4):
    print(f'Cuando usamos {index}:')
    features_upsampled, target_upsampled = upsample(features_train, target_train, index)
    BestF1AndAucRoc(features_upsampled, target_upsampled, features_valid, target_valid)

#### Conclusion

We conducted experiments using both oversampling techniques and class weight adjustments, evaluating performance across RandomForest and LogisticRegression models.

For the RandomForest model, we tuned the number of trees (n_estimators) and the maximum depth (max_depth). For the LogisticRegression model, we optimized the decision threshold to improve classification performance.

Our best result was achieved using the RandomForest model with adjusted class weights. The optimal configuration was:

Best F1: Depth = 6 | Trees = 40 | F1 = 0.624

## Final test

Depth = 6 | Trees = 40 | class_weight = balanced

In [ ]:
final_model = RandomForestClassifier(random_state=12345, n_estimators=40, max_depth=6, class_weight = 'balanced')
final_model.fit(features_train, target_train)
predicted_test = final_model.predict(features_test)
f1_test = f1_score(target_test, predicted_test)
aucroc_test = roc_auc_score(target_test, predicted_test)
print(f'F1_score = {f1_test}')
print()
print(f'AucRoc = {aucroc_test}')

## Conclusion

First, we conducted an exploratory analysis to determine which features were relevant to our model and which were not. Based on this assessment, we removed the non-essential columns and applied label encoding to properly handle the categorical variables.

Next, we examined the presence of class imbalance. Our analysis confirmed a significant imbalance, with an approximate ratio of 3:1, where class 0 was the majority class.

We then implemented both RandomForest and LogisticRegression models to evaluate performance using the F1-score without balancing the dataset. The best result obtained under these conditions was:

Best F1: Depth = 13 | Trees = 40 | F1 = 0.584

To address the class imbalance, we adjusted the class weights using the hyperparameter class_weight = 'balanced'. This approach led to improved model performance.

The final selected model and its optimal hyperparameters were:

final_model = RandomForestClassifier(random_state=12345, n_estimators=40, max_depth=6, class_weight='balanced')

The corresponding performance metrics were:

F1-score = 0.5935613682092555
ROC-AUC = 0.7611918371507681